In [23]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

In [24]:
import pandas as pd

train = pd.read_csv("train_cleaned.csv")

In [25]:
# Pisahkan fitur dan target
X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

# Split internal: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [26]:
# Kolom kategorikal dan numerik
categorical_features = ["gender", "academic_work_impact"]

numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

# Preprocessing yang sama untuk seluruh model
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

# 10-fold CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [27]:
# Model dan parameter dasar. ROC-AUC = 0.96051
models = {
    "HistGradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=300,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}
model = models['HistGradient Boosting']
print(model)


HistGradientBoostingClassifier(l2_regularization=1.0, learning_rate=0.08,
                               max_iter=300, random_state=42)


In [28]:
pipeline = Pipeline([
				("preprocessor", preprocessor),
				("model", model)
		])

In [29]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, randint

In [30]:
# Pipeline baru khusus untuk tuning
tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True
    ))
])

In [31]:
# Rentang parameter yang akan dicoba
param_distributions = {
    # Seberapa besar langkah belajar model
    "model__learning_rate": loguniform(0.02, 0.15),

    # Jumlah maksimal tahap boosting
    "model__max_iter": randint(200, 701),

    # Kompleksitas setiap pohon
    "model__max_leaf_nodes": randint(15, 64),

    # Minimum jumlah data pada daun/node akhir
    "model__min_samples_leaf": randint(10, 51),

    # Regularisasi untuk menahan overfitting
    "model__l2_regularization": loguniform(0.0001, 10.0)
}

In [ ]:
# Random search: mencoba kombinasi parameter secara acak
random_search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=12,                 # 12 kombinasi parameter
    scoring="roc_auc",
    cv=cv,                     # StratifiedKFold 10-fold
    n_jobs=2,                  # Naikkan bila RAM/CPU aman
    verbose=3,
    random_state=42,
    refit=True,                # setelah menemukan terbaik, fit ulang pada seluruh X_train
    return_train_score=True 
)

In [33]:
# Mulai tuning
random_search.fit(X_train, y_train)

print("\nParameter terbaik:")
print(random_search.best_params_)

print(f"\nROC-AUC CV terbaik: {random_search.best_score_:.5f}")

Fitting 10 folds for each of 12 candidates, totalling 120 fits

Parameter terbaik:
{'model__l2_regularization': np.float64(5.5517216852447255), 'model__learning_rate': np.float64(0.13996426971690923), 'model__max_iter': 441, 'model__max_leaf_nodes': 23, 'model__min_samples_leaf': 35}

ROC-AUC CV terbaik: 0.96201


In [34]:
# Ambil model dan parameter terbaik
best_pipeline = random_search.best_estimator_
best_params = random_search.best_params_


In [35]:
best_pipeline = random_search.best_estimator_

In [36]:
from sklearn.metrics import roc_auc_score

y_test_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("ROC-AUC internal test:",
      roc_auc_score(y_test, y_test_prob))

ROC-AUC internal test: 0.9612290834693958


In [37]:
import pandas as pd
import joblib

pd.DataFrame(random_search.cv_results_).sort_values(
    "rank_test_score"
).to_csv("random_search_results.csv", index=False)

joblib.dump(best_pipeline, "histgradient_tuned.joblib")

import json

with open("best_params.json", "w") as file:
    json.dump(random_search.best_params_, file, indent=4)

print("Hasil tuning dan model terbaik berhasil disimpan.")

Hasil tuning dan model terbaik berhasil disimpan.


In [38]:
# test = pd.read_csv('test_cleaned.csv')

# test_id = test['id']
# Xfinaltest = test.drop(columns=['id'])


In [39]:
# # Probabilitas seseorang masuk kelas addicted_label = 1
# y_prob = pipeline.predict_proba(Xfinaltest)[:, 1]

# # Label final berdasarkan threshold default 0.5
# y_pred = pipeline.predict(Xfinaltest)

In [40]:
# submission = pd.DataFrame({
#     "id": test_id,
#     "addicted_label": y_prob
# })

# submission.to_csv("submission.csv", index=False)